## Thematic Analysis of NHS Staff Survey Responses

First, we need to get a list of global themes for the groups at large. For this, we need to use a long-context LLM with a token window that exceeds the open text responses. The new GPT-4.1-mini has a context length of 1 million tokens (approximately 750,000 words) making it very well suited here.

## Section 1: Create a unified data frame

Currently, the online tool containing survey responses requires you to export one cut at a time (e.g. pay band, disability, etc.). Our first task is to create a unified dataframe that takes the various cuts of data as columns.

In [1]:
import os
import glob
import pandas as pd
from functools import reduce

# Point to directory of CSVs
data_dir = "data/"
output_dir = "data/"
os.makedirs(output_dir, exist_ok=True)

# 2) Set output path and check if file exists
out_path = os.path.join(output_dir, "merged_survey.csv")
if os.path.exists(out_path):
    raise FileExistsError(f"⚠️ The file '{out_path}' already exists. Aborting to avoid overwrite.")

# 3) Find all CSVs
csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

# 4) Read & rename each to [group_name, 'Comment']
dfs = []
for fn in csv_files:
    group_name = os.path.splitext(os.path.basename(fn))[0]
    df = pd.read_csv(fn)

    # assume first col = group, second = comment
    df = df.iloc[:, :2].copy()
    df.columns = [group_name, "Comment"]
    dfs.append(df)

# 5) Outer-merge them all on "Comment"
merged = reduce(
    lambda left, right: pd.merge(left, right, on="Comment", how="outer"),
    dfs
)

# 6) Fill missing with "N/A"
merged = merged.fillna("N/A")

# 7) Save to CSV
merged.to_csv(out_path, index=False, encoding="utf-8")

print(f"✅ Unified DataFrame with shape {merged.shape} saved to {out_path}")


✅ Unified DataFrame with shape (1005, 11) saved to data/merged_survey.csv


In [2]:
merged.head()

,occupation_group,Comment,lgbtq,disability,age,service_line,division,gender,payband,staff_group,bme
0,Support to Allied Health Professionals,A big source of dissatisfaction within my role...,Heterosexual or Straight,No Disability,21-30,"CYP, Families and Sefton",Community Care Division,Male,Band 4,Additional Clinical Services,White
1,Nursing auxiliary / Nursing assistant / Health...,A four day week would be help with health and ...,Heterosexual or Straight,No Disability,51-65,"CYP, Families and Sefton",Community Care Division,Female,Band 3,Additional Clinical Services,White
2,Admin & Clerical,A place of work to hot desk from. Working from...,Heterosexual or Straight,No Disability,31-40,Patient Safety,N/A,Female,Band 4,Administrative and Clerical,White
3,Learning disabilities,A positive organisation that does the best pos...,Heterosexual or Straight,No Disability,51-65,Executive Nursing,N/A,Female,N/A,Nursing and Midwifery Registered,White
4,Not available,Admin staff are being forced to work in the of...,Heterosexual or Straight,Disability,51-65,Liverpool Community,Community Care Division,Female,Band 3,Administrative and Clerical,White


Now that we have our unified dataset, we can do some exploratory data analysis.

In [ ]:
from ydata_profiling import ProfileReport

# assume `merged` is your DataFrame
profile = ProfileReport(merged, 
                        title="Merged Survey Data Profile", 
                        explorative=True)

# In a Jupyter notebook in VS Code
profile.to_notebook_iframe()


# Or generate a standalone HTML report:
#profile.to_file("merged_survey_profile.html")


In [3]:
# Extract just the Comments as a single string
themes = "\n".join(merged['Comment'].dropna())
type(themes)

str

In [15]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError
from typing import List

# Activate OpenAI API Key
load_dotenv('api.env')
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Develop prompt for global theme synthesis
prompt = f"""
You are an expert in qualitative analysis and topic modelling for internal staff surveys, with the goal of supporting organisational change.

Given the following anonymised NHS staff survey responses, identify latent topics or themes that emerge organically from the data. Avoid simply repeating comments; instead, synthesise underlying patterns, concepts, concerns, or sentiments expressed by staff.

Ensure that the themes you identify are fair, logical, and consistent, as I will be tabulating this information for further analysis.

Group similar ideas together unless there are clear, meaningful differences. Only split into separate themes when distinct patterns, concerns, or experiences emerge.

Keep the number of themes proportionate to the diversity of the staff comments: if many comments express similar ideas, consolidate them into broader themes; if comments express genuinely different ideas, capture those differences.

Your themes should be blind to sentiment; there should be no good or bad themes. For example, there shouldn't be a "Positive team dynamics and support" theme. Instead, there should be a "Team dynamics/support" theme which holds both positive and negative responses.

Do not use nested or sub-themes.

Return your output strictly as a JSON object with a single key "themes", which maps to a list of objects. Each object must have two keys:
- "name": the concise theme name (string)
- "description": a very brief, 1–2 sentence description of what the theme covers (string)

Here are the survey responses:

{themes}
"""

# Define Pydantic model
class Theme(BaseModel):
    name: str
    description: str

class ThemeList(BaseModel):
    themes: List[Theme]

# Call the model
response = client.responses.create(
    model="gpt-4.1",
    input=prompt
)

# Clean and validate the model's output
text = response.output_text.strip()

# Remove markdown ```json ``` wrappers if present
if text.startswith("```json"):
    text = text.removeprefix("```json").strip()
if text.endswith("```"):
    text = text.removesuffix("```").strip()

# Parse and validate the output
try:
    parsed = ThemeList(**json.loads(text))
    num_themes = len(parsed.themes)
    print(f"✅ GPT-4.1 has identified {num_themes} themes. They are:\n")
    for theme in parsed.themes:
        print(f"- {theme.name}: {theme.description}")
except ValidationError as e:
    print("Validation failed:", e)

# Save
with open('metadata/themes_with_descriptions.json', 'w', encoding='utf-8') as f:
     f.write(text)

✅ GPT-4.1 has identified 21 themes. They are:

- Staffing Levels and Workload: Comments related to the sufficiency of staffing, workload demands, overtime, coverage, burnout, and resource allocation to support effective patient care and manageable staff duties.
- Leadership and Management Practices: Feedback about the behaviours, visibility, competence, and decision-making of all levels of management, including supportiveness, approachability, experience, and the ability to address or escalate issues.
- Career Development and Progression: Recurring points on the availability, fairness, transparency, and accessibility of career advancement opportunities, training, support for further education or qualifications, and discussion of banding structures.
- Pay and Financial Reward: Perspectives on pay, equity of pay across roles and bands, adequacy of salaries relative to role complexity and market, cost of living, and related benefits/rewards.
- Flexible, Hybrid and Remote Working: Observat

In [6]:
# Export themes
with open("metadata/themes.json", "w", encoding="utf-8") as f:
    json.dump(parsed.themes, f, ensure_ascii=False, indent=2)


## Section 2: Thematic Coding

Now that we have our global themes, we can assign them to our survey responses. We will use a lighter-weight model (GPT-4o-mini) to output Yes or No for each of our themes.

In [18]:
from pydantic import create_model, ValidationError
from typing import Literal, Dict
from tqdm import tqdm
import json
import pandas as pd

# Extract comments from `merged` df
comments = merged['Comment']

# Sample just 10 comments for testing
sampled_indices = comments.sample(n=10, random_state=42).index
comments_sample = comments.loc[sampled_indices]
merged_sample = merged.loc[sampled_indices].reset_index(drop=True)

# Create a dynamic Pydantic model with one field per theme
fields = {theme.name: (Literal["Yes", "No"], ...) for theme in parsed.themes}
ThemeAssignment = create_model('ThemeAssignment', **fields)

# Function to query GPT-4o-mini for a single comment
def assign_themes(comment_text: str) -> Dict[str, str]:
    prompt = f"""
You are an expert in qualitative analysis for internal staff surveys.

Given the following survey comment:

\"{comment_text}\"

Assign "Yes" or "No" to each of the following themes based on whether the comment clearly relates to it. Be strict: only assign "Yes" if the theme is explicitly relevant.

Return your response strictly as a JSON object with each theme as a key and "Yes" or "No" as the value. No commentary.

Themes:
{[theme.name for theme in parsed.themes]}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    text = response.output_text.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json").strip()
    if text.endswith("```"):
        text = text.removesuffix("```").strip()

    try:
        json_data = json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed for comment: {comment_text[:50]}... \nError: {e}")
        return {theme.name: "No" for theme in parsed.themes}

    try:
        validated = ThemeAssignment(**json_data)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Pydantic validation failed for comment: {comment_text[:50]}... \nError: {e}")
        return {theme.name: "No" for theme in parsed.themes}

# Iterate over sample comments and assign themes with tqdm progress bar
all_assignments = []
for comment in tqdm(comments_sample, desc="Assigning themes to sample comments"):
    assignment = assign_themes(comment)
    all_assignments.append(assignment)

# Convert to DataFrame
theme_assignments_df = pd.DataFrame(all_assignments)

# ✅ Append results to full sample (not just Comment column)
merged_sampled = pd.concat([merged_sample.reset_index(drop=True), theme_assignments_df], axis=1)

print(f"✅ Finished assigning themes for sample! Final shape: {merged_sampled.shape}")

merged_sampled.head(n=10)


Assigning themes to sample comments: 100%|██████████| 10/10 [00:51<00:00,  5.11s/it]

✅ Finished assigning themes for sample! Final shape: (10, 32)


,occupation_group,Comment,lgbtq,disability,age,service_line,division,gender,payband,staff_group,...,Work Environment and Facilities,Team Dynamics and Peer Support,Recognition and Feedback,Patient Care and Service Quality,"Recruitment, Onboarding and Retention","Sickness, Absence, and Attendance Management","Equality, Diversity and Inclusion",Administrative and Non-Clinical Roles,Systems and Digital Transformation,Trust Size and Organisational Structure
0,Admin & Clerical,This trust would run a lot smoother if they we...,Heterosexual or Straight,No Disability,21-30,Workforce,N/A,Female,Band 5,Administrative and Clerical,...,No,No,No,No,Yes,No,No,No,No,Yes
1,Central Functions / Corporate Services,Our team has been undergoing re- structure for...,Heterosexual or Straight,No Disability,41-50,N/A,N/A,Female,Band 6,Administrative and Clerical,...,No,No,No,No,No,No,No,No,No,No
2,Psychotherapy,Senior management within the NHS Talking thera...,Heterosexual or Straight,No Disability,31-40,Dental and Mid Mersey,Community Care Division,Male,Band 7,Add Prof Scientific and Technic,...,No,No,No,Yes,Yes,No,No,No,No,No
3,Other qualified Allied Health Professionals,Long standing staffing issues have been acknow...,Heterosexual or Straight,No Disability,51-65,Liverpool Community,Community Care Division,Female,Band 7,Allied Health Professionals,...,No,No,No,No,Yes,No,No,No,No,No
4,N/A,"I have worked for NHS for many years, but seve...",Heterosexual or Straight,No Disability,66+,Dental and Mid Mersey,Community Care Division,Female,Band 2,Administrative and Clerical,...,No,No,Yes,No,Yes,No,No,No,No,No
5,Other qualified Allied Health Professionals,Podiatry are sometimes not considered as a hig...,Heterosexual or Straight,Not available,66+,"CYP, Families and Sefton",Community Care Division,Female,Band 4,Additional Clinical Services,...,No,Yes,No,Yes,No,No,No,No,No,No
6,Admin & Clerical,"Poor direction, no scope for development, no c...",Heterosexual or Straight,No Disability,51-65,Workforce,N/A,Male,Band 3,Administrative and Clerical,...,No,No,No,No,No,No,No,No,No,No
7,Mental health,Mersey Care has undone a lot of the good work ...,Heterosexual or Straight,No Disability,51-65,Mental Health Community,N/A,Male,Band 6,Nursing and Midwifery Registered,...,No,No,No,Yes,Yes,No,No,No,No,No
8,Mental health,I have been a ward manager for [number removed...,Heterosexual or Straight,Disability,31-40,Offender Health & Forensic Community,N/A,Female,Band 7,Nursing and Midwifery Registered,...,No,No,No,Yes,Yes,No,No,Yes,No,No
9,Admin & Clerical,Difficult survey to complete due to situation ...,Heterosexual or Straight,No Disability,41-50,Liverpool Community,Community Care Division,Female,Band 3,Administrative and Clerical,...,No,Yes,Yes,No,No,No,No,Yes,No,Yes


Now that we've done this for our sample, we can repeat for the full dataset.

In [20]:
# Run theme assignment on the full dataset (not just sample)
all_assignments_full = []
for comment in tqdm(merged['Comment'], desc="Assigning themes to full dataset"):
    assignment = assign_themes(comment)
    all_assignments_full.append(assignment)

# Convert to DataFrame
theme_assignments_full_df = pd.DataFrame(all_assignments_full)

# Concatenate with original merged dataframe
merged_assigned_full = pd.concat([merged.reset_index(drop=True), theme_assignments_full_df], axis=1)

print(f"✅ Finished assigning themes to full dataset! Final shape: {merged_assigned_full.shape}")

merged_assigned_full.to_csv('data/complete.csv')

merged_assigned_full.head()


Assigning themes to full dataset: 100%|██████████| 1005/1005 [1:32:49<00:00,  5.54s/it]

✅ Finished assigning themes to full dataset! Final shape: (1005, 32)


,occupation_group,Comment,lgbtq,disability,age,service_line,division,gender,payband,staff_group,...,Work Environment and Facilities,Team Dynamics and Peer Support,Recognition and Feedback,Patient Care and Service Quality,"Recruitment, Onboarding and Retention","Sickness, Absence, and Attendance Management","Equality, Diversity and Inclusion",Administrative and Non-Clinical Roles,Systems and Digital Transformation,Trust Size and Organisational Structure
0,Support to Allied Health Professionals,A big source of dissatisfaction within my role...,Heterosexual or Straight,No Disability,21-30,"CYP, Families and Sefton",Community Care Division,Male,Band 4,Additional Clinical Services,...,No,No,No,No,No,No,No,No,No,No
1,Nursing auxiliary / Nursing assistant / Health...,A four day week would be help with health and ...,Heterosexual or Straight,No Disability,51-65,"CYP, Families and Sefton",Community Care Division,Female,Band 3,Additional Clinical Services,...,No,No,No,No,No,No,No,No,No,No
2,Admin & Clerical,A place of work to hot desk from. Working from...,Heterosexual or Straight,No Disability,31-40,Patient Safety,N/A,Female,Band 4,Administrative and Clerical,...,Yes,Yes,No,No,No,No,No,No,No,No
3,Learning disabilities,A positive organisation that does the best pos...,Heterosexual or Straight,No Disability,51-65,Executive Nursing,N/A,Female,N/A,Nursing and Midwifery Registered,...,No,No,No,Yes,No,No,No,No,No,No
4,Not available,Admin staff are being forced to work in the of...,Heterosexual or Straight,Disability,51-65,Liverpool Community,Community Care Division,Female,Band 3,Administrative and Clerical,...,No,No,Yes,No,No,No,Yes,Yes,No,No


## Section 3: Extract Meta-Tags

In addition to themes, it would be useful to be able to filter the survey comments by (a) whether or not the comment contains an actionable suggestion, (b) whether or not the comment is urgent or time-bound, (c) whether the comment contains positive sentiment, (d) whether the comment contains negative sentiment.

In [28]:
class MetaLabels(BaseModel):
    suggestion: Literal["Yes", "No"]
    urgent: Literal["Yes", "No"]
    positive: Literal["Yes", "No"]
    negative: Literal["Yes", "No"]
    
def assign_meta_labels(comment_text: str) -> dict:
    prompt = f"""
You are an expert in analysing internal NHS staff survey responses.

Given the following staff comment, label the presence of the following features using "Yes" or "No" only:

1. **suggestion**: Does the comment contain a **tangible** suggestion, proposal, or ask that could, in theory, be easily actioned? Purely negative comments that do not lend themselves to precise actionability are **not** suggestions. A suggestion **must** also be precise and **not** vague or general
2. **urgent**: Does the comment contain an urgent or time-sensitive concern which is **serious**? Long-standing issues are likely not urgent.
3. **positive**: Does the comment contain praise, appreciation, or a generally positive sentiment?
4. **negative**: Does the comment contain criticism, complaint, or a generally negative sentiment?

Be strict and consistent in your interpretation. None of the above flags are mutually exclusive (i.e. something can contain both positive and negative sentiment, or can be a suggestion and urgent).

Equally, it is perfectly acceptable if the comment reflects none of the above tags.

Output only a valid JSON object for each of the above flags.

Comment:
\"{comment_text}\"
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    text = response.output_text.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json").strip()
    if text.endswith("```"):
        text = text.removesuffix("```").strip()

    try:
        json_data = json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed for comment: {comment_text[:50]}... \nError: {e}")
        return {"suggestion": "No", "urgent": "No", "positive": "No", "negative": "No"}

    try:
        validated = MetaLabels(**json_data)
        return validated.model_dump()
    except ValidationError as e:
        print(f"Validation failed for comment: {comment_text[:50]}... \nError: {e}")
        return {"suggestion": "No", "urgent": "No", "positive": "No", "negative": "No"}

meta_assignments = []
for comment in tqdm(merged['Comment'], desc="Assigning meta labels"):
    labels = assign_meta_labels(comment)
    meta_assignments.append(labels)

meta_labels_df = pd.DataFrame(meta_assignments)
merged_assigned_tagged_full = pd.concat([merged_assigned_full, meta_labels_df], axis=1)

print("✅ Meta labels added!")


Assigning meta labels: 100%|██████████| 1005/1005 [23:05<00:00,  1.38s/it]

✅ Meta labels added!


In [29]:
merged_assigned_tagged_full.head()

,occupation_group,Comment,lgbtq,disability,age,service_line,division,gender,payband,staff_group,...,"Recruitment, Onboarding and Retention","Sickness, Absence, and Attendance Management","Equality, Diversity and Inclusion",Administrative and Non-Clinical Roles,Systems and Digital Transformation,Trust Size and Organisational Structure,suggestion,urgent,positive,negative
0,Support to Allied Health Professionals,A big source of dissatisfaction within my role...,Heterosexual or Straight,No Disability,21-30,"CYP, Families and Sefton",Community Care Division,Male,Band 4,Additional Clinical Services,...,No,No,No,No,No,No,No,No,No,Yes
1,Nursing auxiliary / Nursing assistant / Health...,A four day week would be help with health and ...,Heterosexual or Straight,No Disability,51-65,"CYP, Families and Sefton",Community Care Division,Female,Band 3,Additional Clinical Services,...,No,No,No,No,No,No,Yes,No,No,No
2,Admin & Clerical,A place of work to hot desk from. Working from...,Heterosexual or Straight,No Disability,31-40,Patient Safety,N/A,Female,Band 4,Administrative and Clerical,...,No,No,No,No,No,No,No,No,Yes,No
3,Learning disabilities,A positive organisation that does the best pos...,Heterosexual or Straight,No Disability,51-65,Executive Nursing,N/A,Female,N/A,Nursing and Midwifery Registered,...,No,No,No,No,No,No,No,No,Yes,No
4,Not available,Admin staff are being forced to work in the of...,Heterosexual or Straight,Disability,51-65,Liverpool Community,Community Care Division,Female,Band 3,Administrative and Clerical,...,No,No,Yes,Yes,No,No,No,No,No,Yes


In [30]:
merged_assigned_tagged_full.to_csv('data/complete_tags2.csv')